# TTAP–MDP v1.0 — resultados ejecutables

Este notebook valida el puente con el TTAP determinista y luego activa tiempos inciertos, indisponibilidad operacional y llegadas dinámicas. PPO y DQN se entrenan con los scripts incluidos después de validar estos resultados base.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import pandas as pd

from ttap_mdp import (
    UncertaintyConfig,
    aggregate_results,
    build_talcahuano_scenario,
    evaluate_policy,
    run_episode,
)
from ttap_mdp.baselines import (
    GreedyOnlinePolicy,
    RandomFeasiblePolicy,
    RollingHorizonPolicy,
)

POLICIES = {
    "Random feasible": RandomFeasiblePolicy,
    "Greedy online": GreedyOnlinePolicy,
    "Rolling-horizon matching": RollingHorizonPolicy,
}

## 1. Escenario Talcahuano

Se conservan 7 nodos de demanda, 6 helicópteros, 30 tareas, servicio de 6 minutos, recuperación de 2 minutos y horizonte de 200 minutos.

In [ ]:
scenario = build_talcahuano_scenario()
scenario_summary = pd.DataFrame([{
    "Scenario": scenario.scenario_id,
    "Demand nodes": len(scenario.nodes) - 1,
    "Helicopters": len(scenario.helicopters),
    "Tasks": len(scenario.tasks),
    "Horizon": scenario.horizon,
    "Recovery": scenario.recovery_time,
}])
scenario_summary

## 2. Puente determinista con el primer paper

La incertidumbre se desactiva. La política Greedy debe reproducir exactamente el beneficio, las tareas completadas, las tareas no atendidas, el retorno final y el tiempo de vuelo del caso publicado.

In [ ]:
greedy_result, greedy_simulator = run_episode(
    scenario,
    GreedyOnlinePolicy(),
    uncertainty=UncertaintyConfig.deterministic(),
    seed=218,
)

incomplete = [
    task_id for task_id, state in greedy_simulator.task_states.items()
    if state.status.value != "completed"
]

bridge = pd.DataFrame([{
    "Benefit": greedy_result.benefit,
    "Completed": greedy_result.completed_tasks,
    "Incomplete": ", ".join(incomplete),
    "Makespan": greedy_result.makespan,
    "Flight time": greedy_result.flight_time,
}])

assert abs(greedy_result.benefit - 0.5930769724427806) < 1e-12
assert greedy_result.completed_tasks == 26
assert incomplete == ["T3", "T11", "T23", "T29"]
assert greedy_result.makespan == 117
bridge

In [ ]:
completed_log = pd.DataFrame([
    {
        "Time": record.time,
        "Helicopter": record.helicopter_id,
        "Task": record.task_id,
        "Node": record.node_id,
        "Reward": record.reward,
    }
    for record in greedy_simulator.log
    if record.event == "task_completed"
])
completed_log.head(10)

## 3. Comparación determinista

Random usa semillas diferentes; Greedy y horizonte móvil son deterministas. Esto entrega una primera referencia de calidad antes del entrenamiento RL.

In [ ]:
deterministic_results = []
for policy_factory in POLICIES.values():
    deterministic_results.extend(evaluate_policy(
        scenario,
        policy_factory,
        uncertainty=UncertaintyConfig.deterministic(),
        seeds=range(30),
    ))

deterministic_table = pd.DataFrame(aggregate_results(deterministic_results))
deterministic_table

## 4. Comparación estocástica

La configuración moderada usa CV de viaje 0.15, probabilidad de indisponibilidad 0.05, recuperación entre 5 y 15 minutos, ventana de llegada de 45 minutos y 35% de tareas inicialmente visibles. Son parámetros experimentales modificables, no estimaciones empíricas.

In [ ]:
stochastic_results = []
for policy_factory in POLICIES.values():
    stochastic_results.extend(evaluate_policy(
        scenario,
        policy_factory,
        uncertainty=UncertaintyConfig.moderate(),
        seeds=range(30),
    ))

stochastic_table = pd.DataFrame(aggregate_results(stochastic_results))
stochastic_table

In [ ]:
plot_data = stochastic_table.sort_values("benefit_mean")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].barh(plot_data["policy"], plot_data["benefit_mean"], xerr=plot_data["benefit_sd"], alpha=0.8)
axes[0].set_xlabel("Mean normalized benefit")
axes[0].set_title("Stochastic policy quality")
axes[1].barh(plot_data["policy"], plot_data["completed_mean"], alpha=0.8)
axes[1].set_xlabel("Mean completed tasks")
axes[1].set_title("Task completion")
plt.tight_layout()
plt.show()

## 5. Gymnasium, PPO y DQN

Tras instalar `pip install -e ".[all]"`, valida el entorno con `python -m ttap_mdp.training.validate_environment`. Entrena MaskablePPO con `python -m ttap_mdp.training.train_ppo --scenario small --stochastic --timesteps 100000` y DQN con `python -m ttap_mdp.training.train_dqn --scenario small --stochastic --timesteps 150000`.

La comparación científica posterior debe entrenar varias semillas, reservar escenarios de evaluación no vistos y reportar media, desviación estándar e intervalos de confianza.